**Runtime → Change runtime type → GPU (T4 or better)** before running.

Research prototype — results are not clinically validated.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/GlucoIris'
import os; os.makedirs(f'{BASE}/weights', exist_ok=True); os.makedirs(f'{BASE}/reports', exist_ok=True)

In [ ]:
!pip -q install timm
# Code bundle: upload ml/colab/glucoiris_ml_code.zip to MyDrive/GlucoIris/ first
!rm -rf /content/ml && unzip -q -o "$BASE/glucoiris_ml_code.zip" -d /content/
!ls /content/ml
!nvidia-smi --query-gpu=name,memory.total --format=csv

## IDRiD lesion segmentation data
Download **A. Segmentation.zip** from IEEE Dataport (IDRiD) and upload it to `MyDrive/GlucoIris/`.

In [ ]:
!unzip -q -o "$BASE/A. Segmentation.zip" -d /content/
!ls "/content/A. Segmentation"

## Train the lesion U-Net (MA, HE, EX, SE)
ResNet34-encoder U-Net, lesion-biased 512px patches from 1024px fundus crops, BCE + Dice. Thresholds are chosen on held-out training images; the 27 test images are evaluated once.

In [ ]:
!cd /content && python ml/train_lesions_idrid.py --data "/content/A. Segmentation" --epochs 80 --iters 60 --batch 8 \
    --size 1024 --patch 512 --out "$BASE/weights/unet_lesions_idrid.pt"

In [ ]:
import json
rep = json.load(open('/content/ml/reports/lesions_idrid.json'))
for c, v in rep['meta']['test_metrics'].items():
    print(f"{c:15s} AUPR {v['aupr']:.3f}  Dice {v['dice']:.3f} @ {v['threshold']}")
!cp /content/ml/reports/lesions_idrid.json "$BASE/reports/"; ls -la "$BASE/weights"

Download `MyDrive/GlucoIris/weights/unet_lesions_idrid.pt` into the project at `ml/weights/`, then restart the backend.